In [7]:
import sys, subprocess

# Ensure required packages are available in this Python environment
for pkg in ['pynwb', 'h5py']:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}…")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from spikeinterface import extractors as se
import pynwb
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

# ── Set the NWB file to inspect ───────────────────────────────────────────── #
NWB_FILE = Path('001641/sub-31/sub-31_ses-LE31-S1-01_ecephys.nwb')
# ─────────────────────────────────────────────────────────────────────────────

assert NWB_FILE.exists(), f"Not found: {NWB_FILE.resolve()}"
print(f"✓  {NWB_FILE}  ({NWB_FILE.stat().st_size / 1e6:.1f} MB)")


✓  001641/sub-31/sub-31_ses-LE31-S1-01_ecephys.nwb  (1.1 MB)


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Inspect NWB file structure
# ══════════════════════════════════════════════════════════════════════════════

def find_neurodata_type(h5group, target, _path=''):
    """Recursively find all HDF5 groups whose neurodata_type attribute == target."""
    found = []
    for key, item in h5group.items():
        full = f"{_path}/{key}" if _path else key
        if isinstance(item, h5py.Group):
            nd = item.attrs.get('neurodata_type', b'')
            if isinstance(nd, bytes): nd = nd.decode()
            if nd == target: found.append(full)
            found.extend(find_neurodata_type(item, target, full))
    return found

with h5py.File(NWB_FILE, 'r') as f:

    # ── ElectricalSeries — what SpikeInterface expects for a recording ─────── #
    es_paths = find_neurodata_type(f, 'ElectricalSeries')
    print(f"ElectricalSeries : {es_paths or '(none — spike-only file, no raw voltage)'}")

    # ── What IS in acquisition ────────────────────────────────────────────── #
    print(f"\nAcquisition ({len(f.get('acquisition', {}))} objects):")
    for k, v in f.get('acquisition', {}).items():
        nd = v.attrs.get('neurodata_type', b'')
        if isinstance(nd, bytes): nd = nd.decode()
        n  = v['data'].shape[0] if 'data' in v else (
             v['timestamps'].shape[0] if 'timestamps' in v else '?')
        print(f"  {k:<24} [{nd}]  n_events={n}")

# ── Units table via pynwb ─────────────────────────────────────────────────── #
print()
with pynwb.NWBHDF5IO(str(NWB_FILE), 'r', load_namespaces=True) as io:
    nwb = io.read()
    print(f"Session description : {nwb.session_description}")
    print(f"Subject             : {nwb.subject}")

    if nwb.units:
        print(f"\nUnits table — {len(nwb.units)} units")
        print(f"  Columns : {list(nwb.units.colnames)}")
        print()
        for i in range(len(nwb.units)):
            uid    = nwb.units.id[i]
            name   = nwb.units['Unit_Name'][i]
            area   = nwb.units['Area'][i]
            elec   = nwb.units['Electrode_ID'][i]
            nrn    = nwb.units['Neuron_ID'][i]
            spikes = nwb.units['spike_times'][i]
            dur    = spikes[-1] - spikes[0] if len(spikes) > 1 else 0
            fr     = len(spikes) / dur if dur > 0 else 0
            print(f"  [{uid}] {name:<12}  area={area}  electrode={elec:>2}  "
                  f"neuron={nrn}  n_spikes={len(spikes):6d}  "
                  f"t=[{spikes[0]:.1f}, {spikes[-1]:.1f}] s  "
                  f"mean_FR={fr:.2f} Hz")

        # Check for waveform/template columns
        wf_cols = [c for c in nwb.units.colnames
                   if any(k in c.lower() for k in ('waveform', 'template', 'mean'))]
        print()
        if wf_cols:
            print(f"  ✓ Waveform columns found: {wf_cols}")
        else:
            print(f"  ⚠  No waveform columns — spike times only (common for DANDI exports).")
    else:
        print("  (no units table in this file)")


ElectricalSeries : (none — spike-only file, no raw voltage)

Acquisition (22 objects):
  SpSensor C               [IntervalSeries]  n_events=1552
  SpSensor L               [IntervalSeries]  n_events=442
  SpSensor R               [IntervalSeries]  n_events=530
  SpTestPhase              [IntervalSeries]  n_events=1040
  SpTone A                 [IntervalSeries]  n_events=1096
  SpTone B                 [IntervalSeries]  n_events=316
  SpTone Err               [IntervalSeries]  n_events=790
  SpTone Rwd               [IntervalSeries]  n_events=622
  ViDispenser              [IntervalSeries]  n_events=622
  ViLamp C                 [IntervalSeries]  n_events=1412
  ViLamp D                 [IntervalSeries]  n_events=1040
  ViLamp L                 [IntervalSeries]  n_events=1040
  ViLamp R                 [IntervalSeries]  n_events=1040
  ViSensor C               [IntervalSeries]  n_events=1552
  ViSensor D               [IntervalSeries]  n_events=566
  ViSensor L               [Interva

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# Load spike sorting with SpikeInterface — and why the error occurs
# ══════════════════════════════════════════════════════════════════════════════
#
# WHY THE ERROR FIRES
# ───────────────────
# SpikeInterface tries to auto-detect sampling frequency from an ElectricalSeries.
# DANDI 001641 is a *spike-only* export: spike times + behavioral events,
# but NO raw voltage trace (no ElectricalSeries).
# SI scans the file, finds 0 ES, and raises:
#   "Multiple ElectricalSeries found … Available options are: []"
# (misleading — it means "can't auto-detect, please supply one").
#
# FIX
# ───
# Supply sampling_frequency AND t_start explicitly.
# SI only skips ES detection when BOTH are provided.
# DANDI 001641 used 30 kHz tetrode acquisition; spike times are in seconds.
# ══════════════════════════════════════════════════════════════════════════════

FS = 30_000.0   # Hz — tetrode sampling rate for this dataset

sorting = se.NwbSortingExtractor(
    str(NWB_FILE),
    sampling_frequency = FS,
    t_start            = 0.0,   # both params required to bypass ES detection
)

print("SpikeInterface sorting loaded ✓")
print(f"  Unit IDs   : {sorting.unit_ids}")
print(f"  Fs         : {sorting.get_sampling_frequency():.0f} Hz")
print(f"  Properties : {sorting.get_property_keys()}")
print()

for uid in sorting.unit_ids:
    train   = sorting.get_unit_spike_train(uid)   # sample indices (int64)
    times_s = train / FS                           # seconds
    name    = sorting.get_unit_property(uid, 'Unit_Name')
    area    = sorting.get_unit_property(uid, 'Area')
    elec    = sorting.get_unit_property(uid, 'Electrode_ID')
    nrn     = sorting.get_unit_property(uid, 'Neuron_ID')
    print(f"  Unit {uid} | {name:<12} | area={area:<4} | electrode={elec:>2} "
          f"| neuron={nrn} | {len(train):6d} spikes "
          f"| t=[{times_s[0]:.1f}, {times_s[-1]:.1f}] s")


SpikeInterface sorting loaded ✓
  Unit IDs   : [0 1 2 3]
  Fs         : 30000 Hz
  Properties : ['Area', 'Electrode_ID', 'Neuron_ID', 'Unit_Name', 'electrodes']

  Unit 0 | Unit0601     | area=NAc  | electrode= 6 | neuron=1 |  27758 spikes | t=[-4.9, 19140.1] s
  Unit 1 | Unit0901     | area=NAc  | electrode= 9 | neuron=1 |   4080 spikes | t=[14.1, 19139.2] s
  Unit 2 | Unit1001     | area=NAc  | electrode=10 | neuron=1 |  10767 spikes | t=[-5.1, 19140.5] s
  Unit 3 | Unit1401     | area=NAc  | electrode=14 | neuron=1 |  25062 spikes | t=[-5.1, 19140.7] s


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Extract spike trains, unit metadata, and behavioral event timings
# ══════════════════════════════════════════════════════════════════════════════

# ── A.  Unit metadata + spike-train statistics ────────────────────────────── #
spike_trains = {}      # uid → np.ndarray  (seconds)
unit_records = []

for uid in sorting.unit_ids:
    samples  = sorting.get_unit_spike_train(uid)
    times    = samples / FS
    spike_trains[uid] = times

    isi     = np.diff(times) * 1000            # ms
    dur     = times[-1] - times[0]
    unit_records.append({
        'unit_id':       int(uid),
        'unit_name':     sorting.get_unit_property(uid, 'Unit_Name'),
        'area':          sorting.get_unit_property(uid, 'Area'),
        'electrode_id':  int(sorting.get_unit_property(uid, 'Electrode_ID')),
        'neuron_id':     int(sorting.get_unit_property(uid, 'Neuron_ID')),
        'n_spikes':      len(times),
        'duration_s':    round(float(dur), 2),
        'mean_fr_hz':    round(float(len(times) / dur), 3) if dur > 0 else 0,
        'median_isi_ms': round(float(np.median(isi)), 2)  if len(isi) > 0 else 0,
        'cv_isi':        round(float(np.std(isi) / np.mean(isi)), 3) if len(isi) > 1 else 0,
    })

units_df = pd.DataFrame(unit_records)
print("── Unit metadata ───────────────────────────────────────────────────────")
print(units_df.to_string(index=False))

# ── B.  Behavioral event timings ─────────────────────────────────────────── #
print("\n── Behavioral events ───────────────────────────────────────────────────")
event_times = {}       # event_name → np.ndarray of onset times (seconds)

with pynwb.NWBHDF5IO(str(NWB_FILE), 'r', load_namespaces=True) as io:
    nwb = io.read()
    for name, ts in nwb.acquisition.items():
        try:
            data   = ts.data[:]
            tstamp = ts.timestamps[:]
            onsets = tstamp[data > 0]          # IntervalSeries: +1=onset, -1=offset
            if len(onsets):
                event_times[name] = onsets
                print(f"  {name:<24} {len(onsets):5d} onsets  "
                      f"t=[{onsets[0]:.2f}, {onsets[-1]:.2f}] s")
        except Exception:
            pass

# ── C.  What is NOT in these files ───────────────────────────────────────── #
print()
print("── What is NOT available ───────────────────────────────────────────────")
print("  ✗  Raw voltage trace (no ElectricalSeries)")
print("  ✗  Spike waveform snippets")
print("  ✗  Waveform templates / mean waveforms")
print()
print("  These files are spike-time + behavioral-event exports.  To obtain")
print("  waveform templates you need the original OSort .mat files, or the")
print("  raw continuous recording + a spike sorter (e.g. Kilosort / OSort).")
print()
print("── What IS available (and ready to use) ────────────────────────────────")
print(f"  ✓  Spike times for {len(units_df)} units ({units_df.n_spikes.sum():,} total spikes)")
print(f"  ✓  Unit identity: Unit_Name, Area (NAc/VP), Electrode_ID, Neuron_ID")
print(f"  ✓  {len(event_times)} behavioral event streams "
      f"({', '.join(list(event_times)[:4])} …)")


── Unit metadata ───────────────────────────────────────────────────────
 unit_id unit_name area  electrode_id  neuron_id  n_spikes  duration_s  mean_fr_hz  median_isi_ms  cv_isi
       0  Unit0601  NAc             6          1     27758    19145.08       1.450         276.10   1.546
       1  Unit0901  NAc             9          1      4080    19125.09       0.213         723.43   2.046
       2  Unit1001  NAc            10          1     10767    19145.62       0.562         114.08   4.921
       3  Unit1401  NAc            14          1     25062    19145.85       1.309         471.33   1.177

── Behavioral events ───────────────────────────────────────────────────
  SpSensor C                 776 onsets  t=[44.68, 19111.95] s
  SpSensor L                 221 onsets  t=[48.24, 19115.37] s
  SpSensor R                 265 onsets  t=[233.81, 18842.32] s
  SpTestPhase                520 onsets  t=[46.53, 19114.24] s
  SpTone A                   548 onsets  t=[45.19, 19112.62] s
  SpTon

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# Aggregate all sessions across all subjects
# ══════════════════════════════════════════════════════════════════════════════

BASE_DIR    = Path('001641')
all_records = []

for subject_dir in sorted(BASE_DIR.glob('sub-*')):
    for nwb_path in sorted(subject_dir.glob('*.nwb')):
        subj_id  = subject_dir.name                               # 'sub-31'
        stem     = nwb_path.stem                                  # 'sub-31_ses-LE31-S1-01_ecephys'
        sess_idx = int(stem.split('-S1-')[1].split('_')[0]) if '-S1-' in stem else 0

        try:
            srt = se.NwbSortingExtractor(
                str(nwb_path),
                sampling_frequency = FS,
                t_start            = 0.0,
            )
            for uid in srt.unit_ids:
                t   = srt.get_unit_spike_train(uid) / FS
                isi = np.diff(t) * 1000
                dur = float(t[-1] - t[0]) if len(t) > 1 else 0.0
                all_records.append({
                    'subject':       subj_id,
                    'session':       sess_idx,
                    'unit_id':       int(uid),
                    'unit_name':     srt.get_unit_property(uid, 'Unit_Name'),
                    'area':          srt.get_unit_property(uid, 'Area'),
                    'electrode_id':  int(srt.get_unit_property(uid, 'Electrode_ID')),
                    'neuron_id':     int(srt.get_unit_property(uid, 'Neuron_ID')),
                    'n_spikes':      len(t),
                    'duration_s':    round(dur, 2),
                    'mean_fr_hz':    round(len(t) / dur, 3) if dur > 0 else 0,
                    'median_isi_ms': round(float(np.median(isi)), 2) if len(isi) else 0,
                    'cv_isi':        round(float(np.std(isi) / np.mean(isi)), 3) if len(isi) > 1 else 0,
                })
        except Exception as exc:
            print(f"  ⚠  {nwb_path.name}: {exc}")

summary_df = pd.DataFrame(all_records)

print(f"Loaded {len(summary_df)} unit-sessions  |  "
      f"{summary_df['subject'].nunique()} subjects  |  "
      f"{summary_df.groupby(['subject','session']).ngroups} sessions total")
print(f"Total spikes across all sessions: {summary_df['n_spikes'].sum():,}")
print()

# Per-subject, per-area breakdown
agg = (summary_df
       .groupby(['subject', 'area'])
       .agg(n_units=('unit_id', 'count'),
            total_spikes=('n_spikes', 'sum'),
            mean_fr=('mean_fr_hz', 'mean'),
            median_cv_isi=('cv_isi', 'median'))
       .round(2))
print(agg.to_string())
print()
print("── Per-unit summary (all sessions) ────────────────────────────────────")
cols = ['subject','session','unit_id','unit_name','area','electrode_id',
        'n_spikes','mean_fr_hz','median_isi_ms','cv_isi']
print(summary_df[cols]
      .sort_values(['subject','session','unit_id'])
      .to_string(index=False))


  ⚠  sub-32_ses-LE32-S1-02_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-32_ses-LE32-S1-11_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-34_ses-LE34-S1-01_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-35_ses-LE35-S1-01_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-35_ses-LE35-S1-10_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-42_ses-LE42-S1-03_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-42_ses-LE42-S1-05_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-44_ses-LE44-S1-01_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-44_ses-LE44-S1-02_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-44_ses-LE44-S1-03_ecephys.nwb: units not found in the NWB file!Available options are: [].
  ⚠  sub-4